# 🌊 Automatização dos Estágios Operacionais — Porto Alegre/RS

Pipeline completo: **coleta (ANA + Open-Meteo + Selenium INMET/Poaclima) → CSV com timestamp → classificação nos 5 estágios → dashboard Dash**.

Pasta do projeto no Drive:
`/content/drive/MyDrive/Colab Notebooks/Automatizacao_Estagios_Contingencia`

> ⚠️ Antes de rodar, garanta que o arquivo `ANA_API_ID_SENHA.txt` (ID na 1ª linha, senha na 2ª) está na pasta do projeto no Drive.

In [ ]:
# ── 1) Montar o Google Drive e apontar para a pasta do projeto ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJETO = '/content/drive/MyDrive/Colab Notebooks/Automatizacao_Estagios_Contingencia'
os.makedirs(PROJETO, exist_ok=True)
os.chdir(PROJETO)
if PROJETO not in sys.path:
    sys.path.insert(0, PROJETO)
print('Pasta atual:', os.getcwd())
print('Arquivos:', os.listdir(PROJETO)[:20])

In [ ]:
# ── 2) Instalar dependências + navegador p/ Selenium (com SMOKE TEST) ──
%pip install -q -r requirements.txt

import shutil, subprocess

def _tem_navegador():
    return shutil.which('google-chrome') or shutil.which('google-chrome-stable') \
        or shutil.which('chromium-browser') or shutil.which('chromium')

# Estratégia A — Google Chrome oficial (.deb), com apt-get update ANTES
if not _tem_navegador():
    print('Instalando Google Chrome (.deb)...')
    !apt-get update -qq
    !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/chrome.deb
    resultado = subprocess.run(['apt-get', 'install', '-y', '-qq', '/tmp/chrome.deb'],
                               capture_output=True, text=True)
    if resultado.returncode != 0:
        print('Chrome .deb falhou:', resultado.stderr[-300:])

# Estratégia B — receita StackOverflow (chromium-chromedriver) como fallback
if not _tem_navegador():
    print('Fallback: chromium-chromedriver (receita StackOverflow)...')
    !apt-get update -qq
    !apt install -y -qq chromium-chromedriver
    !cp -f /usr/lib/chromium-browser/chromedriver /usr/bin 2>/dev/null || true

print('Navegador encontrado:', _tem_navegador() or 'NENHUM ✗')

# ── SMOKE TEST: inicia o Chrome em modo HEADLESS (100% invisível —
#    NENHUMA janela abre; ele roda em segundo plano) ──
try:
    from coleta.webdriver_utils import criar_driver
    d = criar_driver()
    d.get('about:blank')
    print('✅ SMOKE TEST OK — Selenium headless funcionando no Colab!')
    d.quit()
except Exception as e:
    print('❌ SMOKE TEST FALHOU:', str(e)[:400])
    print('→ Tente Ambiente de execução > Reiniciar sessão e rode esta célula de novo.')

In [ ]:
# ── 3) Credenciais da ANA: Secrets/env primeiro, TXT como fallback ──
# RECOMENDADO: use os Secrets do Colab (ícone de chave 🔑 na barra lateral):
#   adicione ANA_IDENTIFICADOR e ANA_SENHA e ative o acesso p/ este notebook.
# O código busca nesta ordem: env vars → Secrets do Colab → ANA_API_ID_SENHA.txt
from coleta.ana_api import ler_credenciais
try:
    ident, _senha = ler_credenciais()
    print(f'Credenciais OK ✔  (identificador: {ident[:3]}***, fonte: secrets/env/txt)')
except Exception as e:
    print('⚠️', e)

In [ ]:
# ── 4) Executar o pipeline completo ──
#     coleta → consolida → exporta dados_poa_YYYYMMDD_HHMM.csv → classifica
from main_pipeline import executar_pipeline

snapshot = executar_pipeline(usar_selenium=True)   # False = pular Selenium
print()
print('ESTÁGIO ATUAL:', snapshot['classificacao']['estagio'])

In [ ]:
# ── 5) Conferir os arquivos exportados (CSV + Excel) ──
import pandas as pd, glob
arquivos = sorted(glob.glob('arquivos_gerados_2026/dados_poa_*.csv'))
if not arquivos:
    print('Nenhum arquivo ainda — rode a célula 4 primeiro.')
else:
    ultimo = arquivos[-1]
    print('CSV  :', ultimo)
    print('Excel:', ultimo.replace('.csv', '.xlsx'))
    display(pd.read_csv(ultimo, encoding='utf-8-sig'))

In [ ]:
# ── 6) Simular cenários (teste da lógica E/OU sem coletar nada) ──
from logica.estagios import (classificar_estagio, IndicadoresNumericos,
                             InputsInfraestrutura)

cenario_maio_2024 = IndicadoresNumericos(
    nivel_guaiba_m=5.30, tendencia_guaiba_48h_m=0.60,
    dias_guaiba_acima_inundacao=3,
    acumulado_obs_24h_mm=90, acumulado_obs_72h_mm=300, acumulado_obs_7d_mm=420,
    previsto_48h_mm=120, dias_chuva_intensa_5d=4, inmet_max_severidade='Vermelho',
)
infra = InputsInfraestrutura(colapso_drenagem_urbana=True, obitos_pelo_evento=True,
                             sobrecarga_sistema_saude=True)
res = classificar_estagio(cenario_maio_2024, infra)
print('Estágio:', res['estagio'])
for j in res['justificativas']:
    print(' •', j)

In [ ]:
# ── 7) Abrir o dashboard Dash inline no Colab ──
# O dashboard NÃO depende do Selenium: ele só lê o snapshot gerado pela
# célula 4. Se a célula 4 terminou (mesmo com avisos do Poaclima), rode esta.
from app import rodar_no_colab
rodar_no_colab(porta=8050)